# HW2 – Using APIs for UniProt and Ensembl
**Author:** Victoria Orlova  

In [1]:
import requests
import json
import re
import pandas as pd
import numpy as np

In [2]:
def http_function(endpoint, **kwargs):
    """
    Generic HTTP GET function that returns the JSON response.
    """
    headers = kwargs.get('headers', {})
    kwargs['headers'] = headers
    response = requests.get(endpoint, **kwargs)
    return response

In [3]:
def get_uniprot(accession):
    """
    Retrieve protein data from UniProt REST API for a single accession.
    Returns a Response object.
    """
    endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"
    return http_function(endpoint, headers={'Accept': 'application/json'})

In [4]:
def uniprot_parse_response(resp):
    """
    Parse UniProt JSON response (requests.Response object) and extract:
    organism, geneInfo, sequenceInfo, type.
    Returns a dictionary with those fields, or {'error': ...} on failure.
    """
    try:
        data = resp.json()
    except Exception as e:
        return {'error': f'Failed to parse JSON: {str(e)}'}

    try:
        organism = data.get('organism', {}).get('scientificName', 'N/A')
        geneInfo = data.get('genes', [])
        seq = data.get('sequence', {})
        sequenceInfo = {
            'value': seq.get('value', ''),
            'length': seq.get('length', 'N/A'),
            'molWeight': seq.get('molWeight', 'N/A'),
            'crc64': seq.get('crc64', 'N/A'),
            'md5': seq.get('md5', 'N/A')
        }
        entry_type = 'protein'
        return {
            'organism': organism,
            'geneInfo': geneInfo,
            'sequenceInfo': sequenceInfo,
            'type': entry_type
        }
    except Exception as e:
        return {'error': f'UniProt parsing failed: {str(e)}'}

In [5]:
def get_ensembl(id):
    """
    Retrieve gene/transcript/protein data from Ensembl REST API.
    Returns a Response object.
    """
    endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
    headers = {'Content-Type': 'application/json'}
    return http_function(endpoint, headers=headers)

In [6]:
def ensembl_parse_response(resp: dict):
    """
    Parse Ensembl JSON response and extract:
    object_type, assembly_name, species, db_type, biotype,
    display_name, id, description, canonical_transcript, source
    """
    
    try:
        resp = resp.json()
    except Exception as e:
        return {'error': f'Failed to parse JSON: {str(e)}'}
    
    try:
        if 'error' in resp:
            return {'error': resp['error']}
        
        return {
            'object_type': resp.get('object_type', 'N/A'),
            'assembly_name': resp.get('assembly_name', 'N/A'),
            'species': resp.get('species', 'N/A'),
            'db_type': resp.get('db_type', 'N/A'),
            'biotype': resp.get('biotype', 'N/A'),
            'display_name': resp.get('display_name', 'N/A'),
            'id': resp.get('id', 'N/A'),
            'description': resp.get('description', 'N/A'),
            'canonical_transcript': resp.get('canonical_transcript', 'N/A'),
            'source': resp.get('source', 'N/A')
        }
    except Exception as e:
        return {'error': f'Ensembl parsing failed: {str(e)}'}

In [7]:
def main(ids):
    """
    Accepts a list of IDs, determines their type (UniProt or Ensembl) using regex,
    fetches and parses data from the appropriate API, and returns a pandas DataFrame.
    If an ID is invalid or an error occurs, an error column is added.
    """
    uniprot_pattern = r'^[A-Z][0-9][A-Z0-9]{3,}[0-9]$'
    ensembl_pattern = r'^ENS[A-Z]*[GTEP][0-9]{11}$'

    records = []

    for raw_id in ids:
        acc = raw_id.strip()
        record = {'ID': acc}

        try:
            if re.match(uniprot_pattern, acc):
                resp = get_uniprot(acc)
                info = uniprot_parse_response(resp)
                if 'error' in info:
                    record['error'] = info['error']
                else:
                    record.update(info)
                    record['geneInfo'] = json.dumps(record['geneInfo'])
                    record['sequenceInfo'] = json.dumps(record['sequenceInfo'])

            elif re.match(ensembl_pattern, acc):
                resp = get_ensembl(acc)
                info = ensembl_parse_response(resp)
                if 'error' in info:
                    record['error'] = info['error']
                else:
                    record.update(info)

            else:
                record['error'] = 'ID format not recognized'

        except requests.exceptions.HTTPError as e:
            error_msg = f'HTTP error: {e.response.status_code}'
            try:
                err_data = e.response.json()
                if 'messages' in err_data:
                    error_msg += ' - ' + ', '.join(err_data['messages'])
                elif 'error' in err_data:
                    error_msg += ' - ' + err_data['error']
            except:
                pass
            record['error'] = error_msg
        except Exception as e:
            record['error'] = f'Unexpected error: {str(e)}'

        records.append(record)

    df = pd.DataFrame(records)
    if 'error' in df.columns:
        cols = [c for c in df.columns if c != 'error'] + ['error']
        df = df[cols]
    return df

## Testing with the example IDs from test_data.txt

In [8]:
get_uniprot('P11473')

<Response [200]>

In [9]:
get_uniprot('helloworld')

<Response [400]>

In [10]:
get_uniprot('helloworld').json()

{'url': 'http://rest.uniprot.org/uniprotkb/helloworld',
 'messages': ["The 'accession' value has invalid format. It should be a valid UniProtKB accession"]}

In [11]:
uniprot_parse_response(get_uniprot('P11473'))

{'organism': 'Homo sapiens',
 'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
      'source': 'HGNC',
      'id': 'HGNC:12679'}],
    'value': 'VDR'},
   'synonyms': [{'value': 'NR1I1'}]}],
 'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
  'length': 427,
  'molWeight': 48289,
  'crc64': 'F95F300D042C4CB7',
  'md5': '0D963ACD4A34674368324EE026023597'},
 'type': 'protein'}

In [12]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [13]:
get_ensembl('helloworld')

<Response [400]>

In [14]:
get_ensembl('helloworld').json()

{'error': "ID 'helloworld' not found"}

In [15]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))

{'object_type': 'Gene',
 'assembly_name': 'GRCm39',
 'species': 'mus_musculus',
 'db_type': 'core',
 'biotype': 'protein_coding',
 'display_name': 'Brca2',
 'id': 'ENSMUSG00000041147',
 'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
 'canonical_transcript': 'ENSMUST00000044620.11',
 'source': 'ensembl_havana'}

In [16]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

,ID,organism,geneInfo,sequenceInfo,type,object_type,assembly_name,species,db_type,biotype,display_name,id,description,canonical_transcript,source,error
0,P11473,Homo sapiens,"[{""geneName"": {""evidences"": [{""evidenceCode"": ...","{""value"": ""MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...",protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Q91XI3,Ictidomys tridecemlineatus,"[{""geneName"": {""value"": ""INS""}}]","{""value"": ""MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...",protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,hello,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ID format not recognized
3,ENSG00000157764,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRAF,ENSG00000157764,"B-Raf proto-oncogene, serine/threonine kinase ...",ENST00000646891.2,ensembl_havana,NaN
4,ENSG00000139618,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRCA2,ENSG00000139618,BRCA2 DNA repair associated [Source:HGNC Symbo...,ENST00000380152.8,ensembl_havana,NaN
